# Notebook 1: Synthetic Dataset Generation

This notebook implements the synthetic data generation pipeline. It will:
1.  Load a source document.
2.  Chunk and index it in FAISS.
3.  Generate `(prompt, completion)` pairs using three different strategies (Search, Tutor-Ask, Tutor-Query).
4.  Judge each pair using an `LLM-as-a-Judge`.
5.  Save only the high-scoring pairs to a compressed file.

In [1]:
# 1. Install Libraries
# Note: For web scraping sources (OpenStax, arXiv, Wikipedia), you may need:
# beautifulsoup4, newspaper3k, or pypdf2
%pip install -q transformers torch langchain-text-splitters sentence-transformers faiss-cpu python-dotenv datasets pyarrow tqdm requests accelerate bitsandbytes newspaper3k beautifulsoup4 pypdf2 huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2. Imports and Setup
import os
import getpass
import json
import random
import re
import numpy as np
import faiss
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from datasets import Dataset
from tqdm.notebook import tqdm

# 3. Load environment
load_dotenv()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.0+cpu
CUDA available: False


In [3]:
# 4. Configuration

# --- Teacher Model (Local) ---
# Choose a powerful local model for generating training data
TEACHER_MODEL_NAME = "meta-llama/Llama-3.1-70B-Instruct"
# Alternative options:
# TEACHER_MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
# TEACHER_MODEL_NAME = "mistralai/Mixtral-8x7B-Instruct-v0.1"

# Teacher model quantization (4-bit to fit on most GPUs)
TEACHER_QUANTIZATION = "4bit"  # Options: "4bit", "8bit", "none"

EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
SCORE_THRESHOLD = 4.5  # As defined in our docs
EXAMPLES_PER_DOC = 500 # Adjust as needed
TOP_K = 3

# --- Document Sources with URLs ---
DOC_SOURCES = [
    # Humanities & Philosophy
    {
        "name": "the_republic",
        "url": "https://www.gutenberg.org/ebooks/1497.txt.utf-8",
        "field": "Philosophy"
    },
    {
        "name": "leviathan",
        "url": "https://www.gutenberg.org/ebooks/3207.txt.utf-8",
        "field": "Political Philosophy"
    },
    {
        "name": "meditations",
        "url": "https://www.gutenberg.org/ebooks/2680.txt.utf-8",
        "field": "Philosophy (Stoicism)"
    },
    # STEM
    {
        "name": "biology_2e",
        "url": "https://openstax.org/books/biology-2e/pages/1-introduction",
        "field": "Biology",
        "type": "openstax"  # Special handling needed
    },
    {
        "name": "college_physics_2e",
        "url": "https://openstax.org/books/college-physics-2e/pages/1-introduction",
        "field": "Physics",
        "type": "openstax"  # Special handling needed
    },
    {
        "name": "attention_is_all_you_need",
        "url": "https://arxiv.org/html/1706.03762v7",
        "field": "Computer Science (AI/ML)",
        "type": "arxiv"  # Special handling needed
    },
    # Social Sciences
    {
        "name": "psychology_2e",
        "url": "https://openstax.org/books/psychology-2e/pages/1-introduction",
        "field": "Psychology",
        "type": "openstax"  # Special handling needed
    },
    {
        "name": "principles_of_economics_3e",
        "url": "https://openstax.org/books/principles-economics-3e/pages/1-introduction",
        "field": "Economics",
        "type": "openstax"  # Special handling needed
    },
    {
        "name": "roman_republic",
        "url": "https://en.wikipedia.org/wiki/Roman_Republic",
        "field": "History",
        "type": "wikipedia"  # Special handling needed
    },
    # Technical Documentation
    {
        "name": "langchain_rag_docs",
        "url": "https://python.langchain.com/v0.2/docs/tutorials/rag/",
        "field": "Computer Science (Software)",
        "type": "web"  # Special handling needed
    },
    {
        "name": "huggingface_transformers_docs",
        "url": "https://huggingface.co/docs/transformers/en/index",
        "field": "Computer Science (Software)",
        "type": "web"  # Special handling needed
    }
]

if not DOC_SOURCES:
    print("Warning: DOC_SOURCES is empty. Please add document paths to continue.")
else:
    print(f"Loaded {len(DOC_SOURCES)} document sources:")
    for doc in DOC_SOURCES:
        print(f"  - {doc['name']} ({doc['field']})")

# Load embedding model
print("\nLoading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding model loaded. Dimension: {embedding_dim}")

Loaded 11 document sources:
  - the_republic (Philosophy)
  - leviathan (Political Philosophy)
  - meditations (Philosophy (Stoicism))
  - biology_2e (Biology)
  - college_physics_2e (Physics)
  - attention_is_all_you_need (Computer Science (AI/ML))
  - psychology_2e (Psychology)
  - principles_of_economics_3e (Economics)
  - roman_republic (History)
  - langchain_rag_docs (Computer Science (Software))
  - huggingface_transformers_docs (Computer Science (Software))

Loading embedding model...
Embedding model loaded. Dimension: 384
Embedding model loaded. Dimension: 384


## 4.4 Hugging Face Authentication

**IMPORTANT:** The teacher model (Llama-3.1-70B-Instruct) is a **gated model**. You need to:

1. **Request Access:**
   - Visit https://huggingface.co/meta-llama/Llama-3.1-70B-Instruct
   - Click "Request Access" and accept Meta's license agreement
   - Wait for approval (usually instant)

2. **Get Your Token:**
   - Go to https://huggingface.co/settings/tokens
   - Create a new token (or use existing one)
   - Copy the token

3. **Authenticate:**
   - Run the cell below to login
   - Paste your token when prompted

**Alternative Models (No Authentication Required):**
If you don't want to wait for access, you can use these unrestricted alternatives:
- `mistralai/Mistral-7B-Instruct-v0.3` (smaller, but no auth needed)
- `microsoft/Phi-3-medium-4k-instruct` (14B, good quality)
- `tiiuae/falcon-40b-instruct` (40B, reasonable quality)

Change `TEACHER_MODEL_NAME` in the configuration cell above if you choose an alternative.

In [4]:
# Hugging Face Login
from huggingface_hub import login

# Option 1: Login interactively (recommended for first time)
# This will prompt you to paste your token
# login()

# Option 2: Login with token directly (if you have it in environment)
import os
token = os.getenv("HF_TOKEN")
if token:
    login(token=token)
else:
    print("HF_TOKEN not found in environment. Using interactive login...")
    login()

print("✓ Successfully authenticated with Hugging Face!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Successfully authenticated with Hugging Face!


In [ ]:
# Load Teacher Model with Quantization
print("="*60)
print("LOADING TEACHER MODEL")
print("="*60)
print(f"Model: {TEACHER_MODEL_NAME}")
print(f"Quantization: {TEACHER_QUANTIZATION}")

# Configure quantization
if TEACHER_QUANTIZATION == "4bit":
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    print("Using 4-bit quantization (NF4)")
elif TEACHER_QUANTIZATION == "8bit":
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
    )
    print("Using 8-bit quantization")
else:
    quantization_config = None
    print("No quantization (full precision)")

# Load model
print("\nLoading model... (this may take a few minutes)")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",  # Automatically distribute across GPUs
    torch_dtype=torch.bfloat16 if quantization_config is None else None,
    trust_remote_code=True,
)

# Load tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_MODEL_NAME,
    trust_remote_code=True,
)
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

print("✓ Teacher model loaded successfully!")
print(f"Model device: {teacher_model.device}")
print(f"Model dtype: {teacher_model.dtype}")

# Print memory usage
if torch.cuda.is_available():
    print(f"\nGPU Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

print("="*60)

LOADING TEACHER MODEL
Model: meta-llama/Llama-3.1-70B-Instruct
Quantization: 4bit
Using 4-bit quantization (NF4)

Loading model... (this may take a few minutes)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

d:\codespace\webSpace\digitalbookllm\documentation\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--meta-llama--Llama-3.1-70B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors.index.json:   0%|          | 0.00/59.6k [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

model-00014-of-00030.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00015-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00016-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

## 4.5 Load Local Teacher Model

This cell loads the teacher model with quantization for efficient inference.

In [ ]:
# 4.6 Document Download Helpers
import urllib.request
import requests
from pathlib import Path

# Create docs directory if it doesn't exist
DOCS_DIR = Path("./docs")
DOCS_DIR.mkdir(exist_ok=True)

def download_gutenberg(url, save_path):
    """Download from Project Gutenberg."""
    print(f"Downloading from Project Gutenberg: {url}")
    urllib.request.urlretrieve(url, save_path)
    
def download_web_content(url, save_path):
    """Download web content (for Wikipedia, docs pages, etc.)."""
    print(f"Downloading web content: {url}")
    try:
        # For web scraping, we need additional libraries
        # This is a placeholder - you may need BeautifulSoup or similar
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Save raw HTML for now - you may want to extract text using BeautifulSoup
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Note: Saved as HTML. You may want to extract clean text.")
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False
    return True

def download_document(doc_info):
    """Download a document based on its type."""
    doc_name = doc_info['name']
    url = doc_info['url']
    doc_type = doc_info.get('type', 'gutenberg')
    
    save_path = DOCS_DIR / f"{doc_name}.txt"
    
    # Skip if already exists
    if save_path.exists():
        print(f"✓ {doc_name}.txt already exists. Skipping download.")
        return str(save_path)
    
    try:
        if doc_type == 'gutenberg' or url.startswith('https://www.gutenberg.org'):
            download_gutenberg(url, save_path)
        elif doc_type in ['openstax', 'arxiv', 'wikipedia', 'web']:
            print(f"⚠ {doc_type} source requires special handling (web scraping).")
            print(f"  Manual download recommended from: {url}")
            print(f"  Save as: {save_path}")
            return None
        else:
            download_web_content(url, save_path)
        
        print(f"✓ Downloaded: {save_path}")
        return str(save_path)
    except Exception as e:
        print(f"✗ Error downloading {doc_name}: {e}")
        return None

# Download all documents
print("="*60)
print("DOWNLOADING DOCUMENTS")
print("="*60)

downloaded_paths = []
for doc_info in DOC_SOURCES:
    path = download_document(doc_info)
    if path:
        downloaded_paths.append(path)
    print()

print(f"\nSuccessfully prepared {len(downloaded_paths)} documents.")
print("\nNote: For OpenStax, arXiv, Wikipedia, and documentation sites,")
print("you'll need to manually download or use web scraping tools.")
print("Consider using:")
print("  - BeautifulSoup4 for HTML parsing")
print("  - newspaper3k for article extraction")
print("  - PyPDF2 for PDF documents from arXiv")

DOWNLOADING DOCUMENTS
✓ the_republic.txt already exists. Skipping download.

✓ leviathan.txt already exists. Skipping download.

✓ meditations.txt already exists. Skipping download.

⚠ openstax source requires special handling (web scraping).
  Manual download recommended from: https://openstax.org/books/biology-2e/pages/1-introduction
  Save as: docs\biology_2e.txt

⚠ openstax source requires special handling (web scraping).
  Manual download recommended from: https://openstax.org/books/college-physics-2e/pages/1-introduction
  Save as: docs\college_physics_2e.txt

✓ attention_is_all_you_need.txt already exists. Skipping download.

⚠ openstax source requires special handling (web scraping).
  Manual download recommended from: https://openstax.org/books/psychology-2e/pages/1-introduction
  Save as: docs\psychology_2e.txt

⚠ openstax source requires special handling (web scraping).
  Manual download recommended from: https://openstax.org/books/principles-economics-3e/pages/1-introductio

## 4.8 Manual Download Instructions

Some sources require manual download or additional tools:

### **OpenStax Books** (Biology 2e, Physics 2e, Psychology 2e, Economics 3e)
1. Visit the book page (URLs provided above)
2. Click "Download" → "PDF" or use their API
3. Convert PDF to text using tools like `pdftotext` or `PyPDF2`
4. Save as `./docs/{book_name}.txt`

**Alternative:** Use the OpenStax API or web scraping with BeautifulSoup to extract chapter text.

### **arXiv Papers** (Attention Is All You Need)
1. Visit https://arxiv.org/abs/1706.03762
2. Click "Download PDF"
3. Convert PDF to text using `pdftotext` or `PyPDF2`:
   ```bash
   pip install PyPDF2
   python -c "import PyPDF2; ..."
   ```
4. Save as `./docs/attention_is_all_you_need.txt`

### **Wikipedia Articles** (Roman Republic)
1. Use the `wikipedia-api` Python package:
   ```bash
   pip install wikipedia-api
   ```
2. Or manually copy the article text from the page
3. Save as `./docs/roman_republic.txt`

### **Documentation Sites** (LangChain, Hugging Face)
1. Use web scraping tools like BeautifulSoup or Scrapy
2. Or use browser extensions to save as text
3. Consider using site-specific APIs if available
4. Save as `./docs/{doc_name}.txt`

**Quick Script for Wikipedia:**
```python
import wikipediaapi
wiki = wikipediaapi.Wikipedia('en')
page = wiki.page('Roman_Republic')
with open('./docs/roman_republic.txt', 'w', encoding='utf-8') as f:
    f.write(page.text)
```

In [ ]:
# 4.9 Check Document Status
print("="*60)
print("DOCUMENT STATUS CHECK")
print("="*60)

download_status = {
    "ready": [],
    "missing": [],
    "total": len(DOC_SOURCES)
}

for doc in DOC_SOURCES:
    doc_name = doc['name']
    expected_path = DOCS_DIR / f"{doc_name}.txt"
    
    if expected_path.exists():
        file_size = expected_path.stat().st_size
        download_status["ready"].append((doc_name, doc['field'], file_size))
    else:
        download_status["missing"].append((doc_name, doc['field'], doc.get('type', 'unknown')))

print(f"\n✅ Ready to Process: {len(download_status['ready'])}/{download_status['total']}")
print("-" * 60)
for name, field, size in download_status["ready"]:
    size_kb = size / 1024
    print(f"  ✓ {name:35s} ({field:20s}) - {size_kb:>8.1f} KB")

if download_status["missing"]:
    print(f"\n⚠️  Need Manual Download: {len(download_status['missing'])}/{download_status['total']}")
    print("-" * 60)
    for name, field, doc_type in download_status["missing"]:
        print(f"  ⚠ {name:35s} ({field:20s}) - Type: {doc_type}")
    print("\n💡 See DOCUMENT_SOURCES.md for download instructions")
else:
    print("\n🎉 All documents are ready!")

print("="*60)

DOCUMENT STATUS CHECK

✅ Ready to Process: 5/11
------------------------------------------------------------
  ✓ the_republic                        (Philosophy          ) -   1215.1 KB
  ✓ leviathan                           (Political Philosophy) -   1225.6 KB
  ✓ meditations                         (Philosophy (Stoicism)) -    415.4 KB
  ✓ attention_is_all_you_need           (Computer Science (AI/ML)) -     39.6 KB
  ✓ roman_republic                      (History             ) -     99.1 KB

⚠️  Need Manual Download: 6/11
------------------------------------------------------------
  ⚠ biology_2e                          (Biology             ) - Type: openstax
  ⚠ college_physics_2e                  (Physics             ) - Type: openstax
  ⚠ psychology_2e                       (Psychology          ) - Type: openstax
  ⚠ principles_of_economics_3e          (Economics           ) - Type: openstax
  ⚠ langchain_rag_docs                  (Computer Science (Software)) - Type: web
  ⚠ hu

## 5. Helper Functions (Loading, RAG, LLM Calls)

In [ ]:
def load_and_index_doc(doc_path):
    """Loads a document, chunks it, and creates a FAISS index."""
    print(f"Processing {doc_path}...")
    
    # Check if file exists
    if not os.path.exists(doc_path):
        print(f"⚠ File not found: {doc_path}")
        return None, None, None
    
    try:
        with open(doc_path, 'r', encoding='utf-8') as f:
            full_document = f.read()
    except Exception as e:
        print(f"Error reading file: {e}")
        return None, None, None
    
    if len(full_document.strip()) < 100:
        print(f"⚠ Document too short or empty. Skipping.")
        return None, None, None
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
    )
    doc_chunks = text_splitter.split_text(full_document)
    
    print(f"Embedding {len(doc_chunks)} chunks...")
    chunk_embeddings = embedding_model.encode(doc_chunks, show_progress_bar=False)
    
    index = faiss.IndexFlatL2(embedding_dim)
    index.add(chunk_embeddings.astype('float32'))
    
    chunk_id_to_text = {i: text for i, text in enumerate(doc_chunks)}
    print("Indexing complete.")
    return index, chunk_id_to_text, doc_chunks

def call_teacher_llm(prompt, temperature=0.7, max_tokens=1024):
    """Call the local teacher model for generation."""
    try:
        # Prepare input
        messages = [{"role": "user", "content": prompt}]
        
        # Apply chat template if available
        if hasattr(teacher_tokenizer, 'apply_chat_template'):
            input_text = teacher_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            # Fallback for models without chat template
            input_text = f"<s>[INST] {prompt} [/INST]"
        
        # Tokenize
        inputs = teacher_tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(teacher_model.device)
        
        # Generate
        with torch.no_grad():
            outputs = teacher_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                top_p=0.95,
                pad_token_id=teacher_tokenizer.eos_token_id,
            )
        
        # Decode
        generated_text = teacher_tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        
        return generated_text.strip()
        
    except Exception as e:
        print(f"Error calling teacher model: {e}")
        return None

def get_retrieved_chunks(query_embedding, index, chunk_map, k, selection=None):
    """Performs a FAISS search and returns de-duplicated chunk text."""
    distances, indices = index.search(query_embedding.astype('float32'), k)
    chunks = []
    for idx in indices[0]:
        retrieved_text = chunk_map.get(idx)
        # De-duplicate against the selection
        if selection and retrieved_text.strip() == selection.strip():
            continue
        chunks.append(retrieved_text)
    return "\n---\n".join(chunks) if chunks else "None"

## 6. Generation & Prompting Strategies

In [ ]:
# --- Prompt Templates from `priority-rag.md` ---

TUTOR_PROMPT_TEMPLATE = """You are an AI study buddy... 
[USER'S SELECTION]
{selected_text}
[END USER'S SELECTION]
[RELATED CONTEXT]
{related_context_str}
[END RELATED CONTEXT]
[USER QUESTION]
{query}
Helpful Answer:"""

SEARCH_PROMPT_TEMPLATE = """You are a helpful AI assistant...
[CONTEXT]
{context_str}
[END CONTEXT]
[USER QUESTION]
{query}
Answer:"""

# --- Generation Strategies ---

def generate_search_mode_pair(index, chunk_map, all_chunks, k=TOP_K):
    """Strategy 1: General Q&A"""
    context_chunk = random.choice(all_chunks)
    question_gen_prompt = f"You are a student. Read the following text and write one specific, insightful question that this text could answer.\n\n[TEXT]\n{context_chunk}\n\n[QUESTION]"
    query = call_teacher_llm(question_gen_prompt, temperature=0.8, max_tokens=100)
    if not query: return None, None
    
    query_embedding = embedding_model.encode([query])
    context_str = get_retrieved_chunks(query_embedding, index, chunk_map, k)
    
    final_prompt = SEARCH_PROMPT_TEMPLATE.format(context_str=context_str, query=query)
    completion = call_teacher_llm(final_prompt, temperature=0.2)
    return final_prompt, completion

def generate_tutor_ask_pair(index, chunk_map, all_chunks, k=TOP_K):
    """Strategy 2: Selection + Question"""
    selected_text = random.choice(all_chunks)
    question_gen_prompt = f"You are a student who just highlighted the following text. Write one question about it that asks for elaboration or synthesis.\n\n[HIGHLIGHTED TEXT]\n{selected_text}\n\n[QUESTION]"
    query = call_teacher_llm(question_gen_prompt, temperature=0.8, max_tokens=100)
    if not query: return None, None
    
    search_query = query + " " + selected_text
    query_embedding = embedding_model.encode([search_query])
    related_context_str = get_retrieved_chunks(query_embedding, index, chunk_map, k, selection=selected_text)
    
    final_prompt = TUTOR_PROMPT_TEMPLATE.format(selected_text=selected_text, related_context_str=related_context_str, query=query)
    completion = call_teacher_llm(final_prompt, temperature=0.2)
    return final_prompt, completion

def generate_tutor_query_pair(index, chunk_map, all_chunks, k=TOP_K):
    """Strategy 3: Selection, No Question (Implied Summarize)"""
    selected_text = random.choice(all_chunks)
    query = "Can you summarize this selection and tell me a bit more about it?"
    
    search_query = query + " " + selected_text
    query_embedding = embedding_model.encode([search_query])
    related_context_str = get_retrieved_chunks(query_embedding, index, chunk_map, k, selection=selected_text)
    
    final_prompt = TUTOR_PROMPT_TEMPLATE.format(selected_text=selected_text, related_context_str=related_context_str, query=query)
    completion = call_teacher_llm(final_prompt, temperature=0.2)
    return final_prompt, completion

## 7. Judging Function

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are an expert evaluator for a RAG system. Your goal is to score a generated answer based on a given prompt (which includes context).

**Scoring Metrics (1-5 scale):**
1.  **Faithfulness (Groundedness):** Is the answer 100% supported by the *provided contexts*? (1=Hallucinated, 5=Perfectly Grounded)
2.  **Query Relevance:** Does the answer *directly and completely* address the `[USER QUESTION]`? (1=Irrelevant, 5=Perfectly Relevant)
3.  **Synthesis Quality:** Does the answer *naturally weave* the `[USER'S SELECTION]` and `[RELATED CONTEXT]`? (1=Robotic, 5=Seamless Synthesis) (Score 5 if not applicable, e.g. Search Mode)
4.  **Clarity & Conciseness:** Is the answer easy to understand and non-robotic? (1=Confusing, 5=Clear and Concise)

**Task:** Evaluate the following `completion` generated for the given `prompt`.

---[PROMPT]--- 
{prompt}
---[END PROMPT]---

---[COMPLETION]---
{completion}
---[END COMPLETION]---

**Output Format:** Provide your scores in a single JSON object.
Example: {{"faithfulness": 5, "relevance": 5, "synthesis": 4, "clarity": 5}}

**JSON Score:**
"""

def judge_response(prompt, completion):
    """Uses the LLM-as-a-Judge to score a response."""
    judge_prompt = JUDGE_PROMPT_TEMPLATE.format(prompt=prompt, completion=completion)
    score_str = call_teacher_llm(judge_prompt, temperature=0.0, max_tokens=200)
    
    try:
        # Extract JSON from potential markdown code blocks
        match = re.search(r'\{.*\}', score_str, re.DOTALL)
        if match:
            score_json = json.loads(match.group(0))
        else:
            score_json = json.loads(score_str)
        
        # Calculate composite score
        synthesis_score = score_json.get('synthesis', 5) # Default to 5 if not present
        total_score = (
            score_json.get('faithfulness', 0) * 0.4 +
            score_json.get('relevance', 0) * 0.3 +
            synthesis_score * 0.15 +
            score_json.get('clarity', 0) * 0.15
        )
        return total_score
    except Exception as e:
        print(f"Error parsing judge response: {e}")
        print(f"Raw judge output: {score_str}")
        return 0 # Failed validation


## 8. Main Processing Loop

In [ ]:
strategies = [
    generate_search_mode_pair,
    generate_tutor_ask_pair,
    generate_tutor_query_pair
]

print("="*60)
print("STARTING DATASET GENERATION")
print("="*60)

# Use only successfully downloaded documents
valid_doc_paths = [p for p in downloaded_paths if p and os.path.exists(p)]
print(f"\nProcessing {len(valid_doc_paths)} valid documents...")

for doc_path in valid_doc_paths:
    doc_name = Path(doc_path).stem  # Get filename without extension
    output_dir = f"./dataset_{doc_name}"
    
    if os.path.exists(output_dir):
        print(f"\n✓ Dataset for {doc_name} already exists. Skipping.")
        continue
    
    print(f"\n{'='*60}")
    print(f"Processing: {doc_name}")
    print(f"{'='*60}")
    
    # Load and index the document
    index, chunk_map, all_chunks = load_and_index_doc(doc_path)
    
    # Skip if loading failed
    if index is None:
        print(f"⚠ Skipping {doc_name} due to loading errors.")
        continue
    
    high_quality_pairs = []
    
    pbar = tqdm(range(EXAMPLES_PER_DOC), desc=f"Generating for {doc_name}")
    for _ in pbar:
        strategy = random.choice(strategies)
        prompt, completion = strategy(index, chunk_map, all_chunks)
        
        if not prompt or not completion:
            continue
            
        score = judge_response(prompt, completion)
        pbar.set_postfix({"last_score": f"{score:.2f}", "saved": len(high_quality_pairs)})
        
        if score >= SCORE_THRESHOLD:
            # Format for SFTTrainer
            high_quality_pairs.append({"prompt": prompt, "completion": completion})
            
    print(f"\n✓ Finished {doc_name}. Saved {len(high_quality_pairs)} high-quality examples.")
    
    # 9. Save Dataset in compressed Arrow format
    if high_quality_pairs:
        dataset = Dataset.from_list(high_quality_pairs)
        dataset.save_to_disk(output_dir)
        print(f"✓ Successfully saved dataset to {output_dir}")
    else:
        print(f"⚠ No high-quality pairs generated for {doc_name}. Nothing to save.")

print("\n" + "="*60)
print("ALL DOCUMENTS PROCESSED")
print("="*60)
print(f"\nGenerated datasets are saved in: ./dataset_*/ directories")

STARTING DATASET GENERATION

Processing 5 valid documents...

Processing: the_republic
Processing docs\the_republic.txt...
Embedding 1640 chunks...
Indexing complete.
Indexing complete.


Generating for the_republic:   0%|          | 0/500 [00:00<?, ?it/s]

Error calling Together AI: Error code: 429 - {"message": "You are on tier Build Tier 1, which offers 0.6 queries and 180000 tokens per minute for this model. Please upgrade to higher tier for higher rate limit at https://api.together.ai/settings/billing.", "type_": "model_rate_limit"}
Error calling Together AI: Error code: 429 - {"message": "You are on tier Build Tier 1, which offers 0.6 queries and 180000 tokens per minute for this model. Please upgrade to higher tier for higher rate limit at https://api.together.ai/settings/billing.", "type_": "model_rate_limit"}
Error calling Together AI: Error code: 429 - {"message": "You are on tier Build Tier 1, which offers 0.6 queries and 180000 tokens per minute for this model. Please upgrade to higher tier for higher rate limit at https://api.together.ai/settings/billing.", "type_": "model_rate_limit"}
Error calling Together AI: Error code: 429 - {"message": "You are on tier Build Tier 1, which offers 0.6 queries and 180000 tokens per minute 

KeyboardInterrupt: 